In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)


In [ ]:
# Task 1: Write your code here:

import seaborn as sns
import matplotlib.pyplot as plt
import os

# 1. Read the dataset Q1_data.csv
# Assuming the file is named Q1_data.csv inside the downloaded path
file_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(file_path)

In [ ]:
# Task 2: Write your code here:

print("--- First 5 Rows ---")
print(df.head())

In [ ]:
# Task 3: Write your code here:
# 3. Display dataset information
print("\n--- Dataset Info ---")
df.info()

In [ ]:
# Task 4: Write your code here:
# 4. Show statistical description
print("\n--- Statistical Description ---")
print(df.describe())

In [ ]:
# Task 5: Write your code here:
# 5. Plot the target distribution (delivery_time)
plt.figure(figsize=(10, 6))
sns.histplot(df['delivery_time'], kde=True, color='skyblue')
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop("Order_ID",axis=1)

In [ ]:
# Task 2: Write your code here:
df.isna().sum()

In [ ]:
df = df.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day','Courier_Experience_yrs','Delivery_Time'])

In [ ]:
df.isna().sum()

In [ ]:
# Task 3: Write your code here:
df.drop_duplicates()

In [ ]:
# Task 4: Write your code here:
categorical_cols= ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

# Using pandas get_dummies for One-Hot Encoding
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# We scale only the features, not the target (delivery_time)
features = df.drop(columns=['Delivery_Time'])
target = df['Delivery_Time']

scaled_features = scaler.fit_transform(features)
df_scaled = pd.DataFrame(scaled_features, columns=features.columns)

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# Splitting features and target
X = df_scaled # Scaled features from Part 2
y = target    # delivery_time

In [ ]:
# Task 2,3,4,5: Write your code here:
# Initialize the model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Define KFold cross-validation
# n_splits=5 is a standard choice for a balance between bias and variance
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Calculate MAE across folds
# Note: scikit-learn uses negative MAE for its scoring API
mae_scores = cross_val_score(
    rf_model, X, y,
    scoring='neg_mean_absolute_error',
    cv=kf
)

# Convert negative scores back to positive
mae_scores = -mae_scores

# 4. Print the averaged score
print(f"--- Model Evaluation ---")
print(f"MAE per fold: {mae_scores}")
print(f"Average MAE across all folds: {mae_scores.mean():.2f} minutes")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'model' is your trained estimator and 'X_train' is your features DataFrame
def plot_feature_importance(model, X_train):
    # Create a Series with feature importances and index as feature names
    importances = pd.Series(model.feature_importances_, index=X_train.columns)

    # Sort importances in descending order
    importances = importances.sort_values(ascending=True)

    plt.figure(figsize=(10, 6))
    importances.plot(kind='barh', color='skyblue')
    plt.title('Feature Importance')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Call the function
plot_feature_importance(model, X_train)

In [ ]:
# Task 2: Write your code here:
import seaborn as sns

def plot_prediction_distribution(y_pred):
    plt.figure(figsize=(10, 6))

    # Using Seaborn for a smoother distribution curve (KDE)
    sns.histplot(y_pred, kde=True, bins=30, color='salmon')

    plt.title('Distribution of Predicted Delivery Times')
    plt.xlabel('Predicted Delivery Time (minutes)')
    plt.ylabel('Frequency')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()

# Assuming 'y_pred' contains your model's predictions
plot_prediction_distribution(y_pred)

In [ ]:
# Task Bonus: Write your code here:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error

# Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

# Assuming X and y are your features and target
for train_index, val_index in kf.split(X):
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # 1. Initialize the models
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    cat_model = CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6, verbose=0)

    # 2. Train both models
    rf_model.fit(X_train, y_train)
    cat_model.fit(X_train, y_train)

    # 3. Get predictions from both
    rf_preds = rf_model.predict(X_val)
    cat_preds = cat_model.predict(X_val)

    # 4. Average the predictions
    ensemble_preds = (rf_preds + cat_preds) / 2
